# Prosit 4 — Métaheuristiques pour le VRP-CDR

**Bloc** : Recherche Opérationnelle — CESI Montpellier  
**Équipe** : RIVET Alexandre, LUU Philippe, AFANE Youcef, [Votre nom]  
**Date** : Avril 2026

---

## Table des matières

1. [Analyse du Prosit — Mots-clés et problématique](#1)
2. [Fondamentaux des métaheuristiques](#2)
   - 2.1 Heuristique vs Métaheuristique
   - 2.2 Intensification vs Diversification
   - 2.3 Classifications des métaheuristiques
3. [Le théorème No Free Lunch (NFL)](#3)
4. [Modélisation du voisinage pour le VRP](#4)
5. [Implémentation des métaheuristiques](#5)
   - 5.1 Hill Climbing (référence)
   - 5.2 Recuit Simulé (Simulated Annealing)
   - 5.3 Recherche Tabou
   - 5.4 Algorithme Génétique
6. [Génération d'instances et plan d'expérience](#6)
7. [Évaluation et comparaison des résultats](#7)
8. [Analyse statistique et conclusions](#8)

---
<a id='1'></a>
## 1. Analyse du Prosit — Mots-clés et problématique

### 1.1 Mots-clés identifiés

| Mot-clé | Définition |
|---------|------------|
| **Métaheuristique** | Stratégie générale de résolution approchée, applicable à une large classe de problèmes d'optimisation, combinant intensification et diversification. |
| **Heuristique gloutonne (Greedy)** | Méthode constructive qui fait un choix localement optimal à chaque étape, sans retour en arrière. |
| **Hill Climbing** | Recherche locale itérative qui accepte uniquement les solutions améliorantes dans le voisinage. Pas de diversification → risque d'optimum local. |
| **Intensification** | Phase d'exploitation : on explore en profondeur une zone prometteuse de l'espace de recherche. |
| **Diversification** | Phase d'exploration : on s'éloigne de la zone courante pour découvrir d'autres régions de l'espace de recherche. |
| **Optimum local** | Solution meilleure que tous ses voisins, mais pas nécessairement la meilleure globale. |
| **Voisinage** | Ensemble des solutions atteignables depuis une solution donnée par une transformation élémentaire (un "mouvement"). |
| **Rugosité du paysage** | Caractérise la densité d'optima locaux dans l'espace de recherche. Un paysage rugueux nécessite plus de diversification. |
| **Méthode par trajectoire** | Métaheuristique qui maintient une seule solution courante et la fait évoluer (ex : recuit simulé, tabou). |
| **Méthode par population** | Métaheuristique qui maintient un ensemble de solutions simultanément (ex : algorithme génétique, colonies de fourmis). |
| **Constructive vs Perturbative** | Constructive = on bâtit la solution pas à pas. Perturbative = on modifie une solution existante. |
| **No Free Lunch (NFL)** | Théorème stipulant qu'aucune métaheuristique n'est universellement supérieure à toutes les autres sur l'ensemble des problèmes. |
| **Borne inférieure** | Valeur minimale théorique de la fonction objectif, utilisée pour évaluer la qualité d'une solution approchée. |
| **Plan d'expérience** | Méthodologie structurée pour tester l'influence de paramètres sur les résultats, en limitant le nombre d'expériences. |

### 1.2 Problématique

**Comment choisir, implémenter et évaluer une métaheuristique adaptée à la résolution approchée du VRP-CDR (Vehicle Routing Problem avec Contraintes de livraison Dynamiques et Retour au dépôt) ?**

### 1.3 Hypothèses

- Le Hill Climbing seul est insuffisant car il ne fait pas de diversification.
- Il faut comparer plusieurs métaheuristiques car le NFL interdit de prédire laquelle sera la meilleure.
- La qualité d'une solution peut être estimée grâce à des bornes inférieures.
- Un plan d'expérience rigoureux est nécessaire pour obtenir des résultats représentatifs.

---
<a id='2'></a>
## 2. Fondamentaux des métaheuristiques

### 2.1 Heuristique vs Métaheuristique

Une **heuristique** est une méthode de résolution approchée spécifique à un problème donné. Elle fournit une solution "raisonnable" rapidement, mais sans garantie d'optimalité.

Une **métaheuristique** est un cadre algorithmique général, applicable à de nombreux problèmes d'optimisation combinatoire. Elle orchestre des mécanismes d'**intensification** et de **diversification** pour parcourir efficacement l'espace de recherche.

### 2.2 Intensification vs Diversification

C'est le dilemme fondamental (aussi appelé *exploitation vs exploration*) :

- **Intensification (exploitation)** : On concentre la recherche autour de la meilleure solution connue. On exploite le voisinage en profondeur. Risque : rester piégé dans un optimum local.

- **Diversification (exploration)** : On force l'algorithme à visiter des régions inexplorées de l'espace de recherche. Risque : perdre du temps dans des zones peu prometteuses.

Le **Hill Climbing** ne fait **que** de l'intensification : il accepte uniquement les voisins améliorants et s'arrête dès qu'il atteint un optimum local. C'est pourquoi Agathe ne le considère pas comme une vraie métaheuristique.

### 2.3 Classifications des métaheuristiques

Il existe plusieurs axes de classification (qui se croisent) :

| Axe | Catégorie 1 | Catégorie 2 |
|-----|-------------|-------------|
| Nombre de solutions | **Trajectoire** (1 solution) | **Population** (ensemble de solutions) |
| Construction | **Constructive** (on bâtit la solution) | **Perturbative** (on modifie une solution) |
| Inspiration | **Physique** (recuit simulé) | **Biologique** (génétique, fourmis) |
| Mémoire | **Sans mémoire** (recuit) | **Avec mémoire** (tabou) |

**Exemples de classification :**

| Métaheuristique | Trajectoire/Population | Const./Perturb. | Mémoire |
|-----------------|----------------------|-----------------|----------|
| Hill Climbing | Trajectoire | Perturbative | Non |
| Recuit Simulé | Trajectoire | Perturbative | Non |
| Recherche Tabou | Trajectoire | Perturbative | Oui (liste tabou) |
| GRASP | Trajectoire | Constructive + Perturbative | Non |
| Algorithme Génétique | Population | Perturbative | Non (implicite via population) |
| Colonies de Fourmis | Population | Constructive | Oui (phéromones) |

---
<a id='3'></a>
## 3. Le théorème No Free Lunch (NFL)

Le théorème **No Free Lunch** (Wolpert & Macready, 1997) établit que :

> Sur l'ensemble de **tous** les problèmes d'optimisation possibles, toutes les métaheuristiques ont en moyenne les **mêmes performances**.

**Conséquence pratique** : Il n'existe pas de métaheuristique "universellement meilleure". Pour un problème donné (comme le VRP-CDR), certaines approches seront plus adaptées que d'autres, mais il faut **tester et comparer** expérimentalement.

C'est la raison pour laquelle Agathe insiste sur :
- L'importance de tester plusieurs métaheuristiques
- La nécessité de consulter la littérature scientifique existante sur le VRP
- La mise en place d'un plan d'expérience rigoureux

---
<a id='4'></a>
## 4. Modélisation du voisinage pour le VRP

Pour appliquer des métaheuristiques à base de voisinage, il faut définir les **mouvements** (transformations élémentaires) qui permettent de passer d'une solution à une solution voisine.

### Représentation d'une solution

Une solution du VRP est une liste ordonnée de clients à visiter. Le véhicule part du dépôt (indice 0), visite les clients dans l'ordre, et revient au dépôt.

Exemple : `[0, 3, 1, 4, 2, 0]` → Le véhicule part du dépôt, visite les clients 3, 1, 4, 2 puis revient au dépôt.

### Opérateurs de voisinage classiques

1. **2-opt** : On inverse un sous-segment de la tournée. On retire deux arêtes et on les reconnecte de manière croisée.
2. **Or-opt** : On déplace un segment de 1, 2 ou 3 clients consécutifs vers une autre position.
3. **Swap** : On échange la position de deux clients.
4. **Insertion** : On retire un client et on le réinsère à une autre position.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
import copy
import time
from collections import defaultdict

# Fixer la graine pour la reproductibilité
random.seed(42)
np.random.seed(42)

print("Imports OK")

### 4.1 Génération d'une instance VRP

In [ ]:
def generer_instance(n_clients, largeur=100, hauteur=100, seed=None):
    """
    Génère une instance de VRP.
    
    Paramètres :
        n_clients : nombre de clients (hors dépôt)
        largeur, hauteur : dimensions de la zone géographique
        seed : graine aléatoire pour la reproductibilité
    
    Retourne :
        coords : tableau (n_clients+1, 2) avec le dépôt en indice 0
        dist_matrix : matrice de distances euclidiennes
    """
    if seed is not None:
        np.random.seed(seed)
    
    # Le dépôt est au centre
    depot = np.array([[largeur / 2, hauteur / 2]])
    # Les clients sont répartis aléatoirement
    clients = np.random.rand(n_clients, 2) * [largeur, hauteur]
    coords = np.vstack([depot, clients])
    
    # Matrice de distances euclidiennes
    n = len(coords)
    dist_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            d = np.linalg.norm(coords[i] - coords[j])
            dist_matrix[i][j] = d
            dist_matrix[j][i] = d
    
    return coords, dist_matrix


# Test : instance avec 20 clients
coords, dist_matrix = generer_instance(20, seed=42)
print(f"Instance générée : {len(coords) - 1} clients + 1 dépôt")
print(f"Dépôt en : {coords[0]}")

In [ ]:
def afficher_tournee(coords, tournee, titre="Tournée VRP", cout=None):
    """
    Affiche graphiquement une tournée VRP.
    """
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    
    # Tracer les arêtes de la tournée
    for i in range(len(tournee) - 1):
        a, b = tournee[i], tournee[i + 1]
        ax.plot([coords[a][0], coords[b][0]], 
                [coords[a][1], coords[b][1]], 'b-', alpha=0.6, linewidth=1.5)
    
    # Tracer les clients
    clients = coords[1:]
    ax.scatter(clients[:, 0], clients[:, 1], c='steelblue', s=80, zorder=5, label='Clients')
    
    # Numéroter les clients
    for i in range(1, len(coords)):
        ax.annotate(str(i), coords[i], textcoords="offset points", 
                    xytext=(5, 5), fontsize=8)
    
    # Tracer le dépôt
    ax.scatter(coords[0][0], coords[0][1], c='red', s=200, marker='s', zorder=6, label='Dépôt')
    
    titre_complet = titre
    if cout is not None:
        titre_complet += f" — Coût : {cout:.2f}"
    ax.set_title(titre_complet, fontsize=14)
    ax.legend()
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# Tournée initiale naïve : ordre naturel
n_clients = len(coords) - 1
tournee_naive = [0] + list(range(1, n_clients + 1)) + [0]
cout_naive = sum(dist_matrix[tournee_naive[i]][tournee_naive[i+1]] for i in range(len(tournee_naive)-1))
afficher_tournee(coords, tournee_naive, "Tournée naïve (ordre naturel)", cout_naive)

### 4.2 Opérateurs de voisinage

In [ ]:
def cout_tournee(tournee, dist_matrix):
    """Calcule le coût total d'une tournée."""
    return sum(dist_matrix[tournee[i]][tournee[i+1]] for i in range(len(tournee) - 1))


def voisin_2opt(tournee):
    """
    Mouvement 2-opt : inverse un sous-segment de la tournée.
    On ne touche pas au dépôt (premier et dernier élément).
    """
    n = len(tournee)
    nouvelle = tournee[:]
    # Choisir deux indices dans la partie clients (indices 1 à n-2)
    i = random.randint(1, n - 3)
    j = random.randint(i + 1, n - 2)
    # Inverser le sous-segment [i, j]
    nouvelle[i:j+1] = reversed(nouvelle[i:j+1])
    return nouvelle


def voisin_swap(tournee):
    """
    Mouvement Swap : échange deux clients de position.
    """
    n = len(tournee)
    nouvelle = tournee[:]
    i = random.randint(1, n - 2)
    j = random.randint(1, n - 2)
    while j == i:
        j = random.randint(1, n - 2)
    nouvelle[i], nouvelle[j] = nouvelle[j], nouvelle[i]
    return nouvelle


def voisin_insertion(tournee):
    """
    Mouvement Insertion : retire un client et le réinsère ailleurs.
    """
    n = len(tournee)
    nouvelle = tournee[:]
    i = random.randint(1, n - 2)
    client = nouvelle.pop(i)
    j = random.randint(1, len(nouvelle) - 1)
    nouvelle.insert(j, client)
    return nouvelle


# Démonstration des opérateurs
print(f"Tournée originale : {tournee_naive}")
print(f"Après 2-opt       : {voisin_2opt(tournee_naive)}")
print(f"Après swap        : {voisin_swap(tournee_naive)}")
print(f"Après insertion   : {voisin_insertion(tournee_naive)}")

### 4.3 Solution initiale par heuristique du plus proche voisin

In [ ]:
def plus_proche_voisin(dist_matrix):
    """
    Heuristique gloutonne du plus proche voisin.
    Construit une tournée en choisissant toujours le client non visité le plus proche.
    """
    n = len(dist_matrix)
    visite = [False] * n
    tournee = [0]  # Départ du dépôt
    visite[0] = True
    
    for _ in range(n - 1):
        courant = tournee[-1]
        meilleur = None
        meilleure_dist = float('inf')
        for j in range(n):
            if not visite[j] and dist_matrix[courant][j] < meilleure_dist:
                meilleur = j
                meilleure_dist = dist_matrix[courant][j]
        tournee.append(meilleur)
        visite[meilleur] = True
    
    tournee.append(0)  # Retour au dépôt
    return tournee


tournee_ppv = plus_proche_voisin(dist_matrix)
cout_ppv = cout_tournee(tournee_ppv, dist_matrix)
print(f"Tournée PPV : coût = {cout_ppv:.2f}")
afficher_tournee(coords, tournee_ppv, "Heuristique du Plus Proche Voisin", cout_ppv)

---
<a id='5'></a>
## 5. Implémentation des métaheuristiques

Nous allons implémenter et comparer 4 approches :
1. **Hill Climbing** — référence (pas une vraie métaheuristique)
2. **Recuit Simulé** — trajectoire, avec diversification stochastique
3. **Recherche Tabou** — trajectoire, avec mémoire
4. **Algorithme Génétique** — population

### 5.1 Hill Climbing

**Principe** : À chaque itération, on génère un voisin. Si le voisin est meilleur, on le garde. Sinon, on ne bouge pas. On s'arrête quand aucune amélioration n'est possible.

**Limite** : Uniquement de l'intensification → piégé au premier optimum local rencontré.

In [ ]:
def hill_climbing(dist_matrix, tournee_init, max_iter=5000, operateur=voisin_2opt):
    """
    Hill Climbing : accepte uniquement les améliorations.
    
    Retourne : (meilleure_tournee, meilleur_cout, historique_couts)
    """
    tournee = tournee_init[:]
    cout = cout_tournee(tournee, dist_matrix)
    historique = [cout]
    
    iterations_sans_amelioration = 0
    
    for i in range(max_iter):
        voisin = operateur(tournee)
        cout_v = cout_tournee(voisin, dist_matrix)
        
        if cout_v < cout:
            tournee = voisin
            cout = cout_v
            iterations_sans_amelioration = 0
        else:
            iterations_sans_amelioration += 1
        
        historique.append(cout)
        
        # Arrêt si stagnation prolongée
        if iterations_sans_amelioration > 500:
            break
    
    return tournee, cout, historique


# Exécution
t0 = time.time()
tournee_hc, cout_hc, hist_hc = hill_climbing(dist_matrix, tournee_ppv)
temps_hc = time.time() - t0

print(f"Hill Climbing : coût = {cout_hc:.2f} (temps = {temps_hc:.3f}s)")
print(f"Amélioration vs PPV : {(cout_ppv - cout_hc) / cout_ppv * 100:.1f}%")
afficher_tournee(coords, tournee_hc, "Hill Climbing (2-opt)", cout_hc)

### 5.2 Recuit Simulé (Simulated Annealing)

**Principe** : Inspiré du recuit en métallurgie. On accepte les solutions améliorantes, mais aussi les solutions dégradantes avec une probabilité qui dépend de la **température** $T$ :

$$P(\text{accepter}) = e^{-\frac{\Delta f}{T}}$$

où $\Delta f = f(\text{voisin}) - f(\text{courant}) > 0$ est la dégradation.

- **Température élevée** → forte diversification (on accepte beaucoup de dégradations)
- **Température basse** → forte intensification (comportement proche du Hill Climbing)

La température diminue progressivement selon un **schéma de refroidissement** (ici géométrique : $T_{k+1} = \alpha \cdot T_k$).

In [ ]:
def recuit_simule(dist_matrix, tournee_init, T_init=100, T_min=0.01, 
                  alpha=0.995, max_iter=10000, operateur=voisin_2opt):
    """
    Recuit Simulé.
    
    Paramètres :
        T_init : température initiale
        T_min  : température finale (critère d'arrêt)
        alpha  : facteur de refroidissement (0 < alpha < 1)
        max_iter : nombre max d'itérations par palier de température
    """
    tournee = tournee_init[:]
    cout = cout_tournee(tournee, dist_matrix)
    
    meilleure_tournee = tournee[:]
    meilleur_cout = cout
    
    T = T_init
    historique = [cout]
    
    iteration = 0
    while T > T_min and iteration < max_iter:
        voisin = operateur(tournee)
        cout_v = cout_tournee(voisin, dist_matrix)
        delta = cout_v - cout
        
        # Critère de Metropolis
        if delta < 0:
            # Amélioration : on accepte toujours
            tournee = voisin
            cout = cout_v
        elif random.random() < np.exp(-delta / T):
            # Dégradation : on accepte avec probabilité exp(-delta/T)
            tournee = voisin
            cout = cout_v
        
        # Mise à jour de la meilleure solution
        if cout < meilleur_cout:
            meilleure_tournee = tournee[:]
            meilleur_cout = cout
        
        # Refroidissement
        T *= alpha
        iteration += 1
        historique.append(meilleur_cout)
    
    return meilleure_tournee, meilleur_cout, historique


# Exécution
t0 = time.time()
tournee_sa, cout_sa, hist_sa = recuit_simule(dist_matrix, tournee_ppv)
temps_sa = time.time() - t0

print(f"Recuit Simulé : coût = {cout_sa:.2f} (temps = {temps_sa:.3f}s)")
print(f"Amélioration vs PPV : {(cout_ppv - cout_sa) / cout_ppv * 100:.1f}%")
afficher_tournee(coords, tournee_sa, "Recuit Simulé (2-opt)", cout_sa)

### 5.3 Recherche Tabou

**Principe** : À chaque itération, on explore le voisinage et on choisit le **meilleur voisin**, même s'il est moins bon que la solution courante (diversification). Pour éviter de revenir en arrière (cycler), on maintient une **liste tabou** qui interdit les mouvements récemment effectués.

- **Intensification** : on choisit le meilleur voisin disponible
- **Diversification** : on accepte les dégradations, la liste tabou empêche le cyclage
- **Critère d'aspiration** : un mouvement tabou est quand même accepté s'il améliore la meilleure solution connue

In [ ]:
def recherche_tabou(dist_matrix, tournee_init, max_iter=2000, 
                    taille_tabou=20, n_voisins=50):
    """
    Recherche Tabou avec mouvement 2-opt.
    
    Paramètres :
        taille_tabou : durée pendant laquelle un mouvement est interdit
        n_voisins : nombre de voisins explorés à chaque itération
    """
    tournee = tournee_init[:]
    cout = cout_tournee(tournee, dist_matrix)
    
    meilleure_tournee = tournee[:]
    meilleur_cout = cout
    
    # Liste tabou : stocke les mouvements (i, j) interdits
    liste_tabou = []
    historique = [cout]
    
    for iteration in range(max_iter):
        meilleur_voisin = None
        meilleur_cout_voisin = float('inf')
        meilleur_mouvement = None
        
        # Explorer n_voisins voisins 2-opt
        n = len(tournee)
        for _ in range(n_voisins):
            i = random.randint(1, n - 3)
            j = random.randint(i + 1, n - 2)
            
            # Générer le voisin 2-opt
            voisin = tournee[:]
            voisin[i:j+1] = reversed(voisin[i:j+1])
            cout_v = cout_tournee(voisin, dist_matrix)
            
            mouvement = (i, j)
            
            # Vérifier si le mouvement est tabou
            est_tabou = mouvement in liste_tabou
            
            # Critère d'aspiration : accepter un tabou s'il améliore le best
            if est_tabou and cout_v >= meilleur_cout:
                continue
            
            if cout_v < meilleur_cout_voisin:
                meilleur_voisin = voisin
                meilleur_cout_voisin = cout_v
                meilleur_mouvement = mouvement
        
        if meilleur_voisin is None:
            break
        
        # Appliquer le meilleur mouvement
        tournee = meilleur_voisin
        cout = meilleur_cout_voisin
        
        # Mettre à jour la liste tabou
        liste_tabou.append(meilleur_mouvement)
        if len(liste_tabou) > taille_tabou:
            liste_tabou.pop(0)
        
        # Mise à jour de la meilleure solution
        if cout < meilleur_cout:
            meilleure_tournee = tournee[:]
            meilleur_cout = cout
        
        historique.append(meilleur_cout)
    
    return meilleure_tournee, meilleur_cout, historique


# Exécution
t0 = time.time()
tournee_tabou, cout_tabou, hist_tabou = recherche_tabou(dist_matrix, tournee_ppv)
temps_tabou = time.time() - t0

print(f"Recherche Tabou : coût = {cout_tabou:.2f} (temps = {temps_tabou:.3f}s)")
print(f"Amélioration vs PPV : {(cout_ppv - cout_tabou) / cout_ppv * 100:.1f}%")
afficher_tournee(coords, tournee_tabou, "Recherche Tabou (2-opt)", cout_tabou)

### 5.4 Algorithme Génétique

**Principe** : Inspiré de l'évolution naturelle. On manipule une **population** de solutions qui évolue par :

- **Sélection** : les meilleures solutions ont plus de chances d'être choisies comme parents
- **Croisement (crossover)** : deux parents produisent un enfant combinant leurs caractéristiques
- **Mutation** : perturbation aléatoire pour maintenir la diversité

**Intensification** : via la sélection (les meilleurs survivent)  
**Diversification** : via la mutation et le croisement

Pour le TSP/VRP, on utilise le croisement **OX (Order Crossover)** qui préserve l'ordre relatif des clients.

In [ ]:
def croisement_ox(parent1, parent2):
    """
    Order Crossover (OX) pour le TSP/VRP.
    On travaille sur la partie clients (sans le dépôt).
    """
    # Extraire les clients (sans dépôt de début et fin)
    p1 = parent1[1:-1]
    p2 = parent2[1:-1]
    n = len(p1)
    
    # Choisir une sous-séquence du parent 1
    i = random.randint(0, n - 2)
    j = random.randint(i + 1, n - 1)
    
    # Copier la sous-séquence du parent 1
    enfant = [None] * n
    enfant[i:j+1] = p1[i:j+1]
    
    # Remplir le reste avec les éléments du parent 2 (dans l'ordre)
    elements_p1 = set(p1[i:j+1])
    pos = (j + 1) % n
    for elem in p2:
        if elem not in elements_p1:
            while enfant[pos] is not None:
                pos = (pos + 1) % n
            enfant[pos] = elem
            pos = (pos + 1) % n
    
    return [0] + enfant + [0]


def mutation(tournee, taux_mutation=0.1):
    """Mutation par swap avec une probabilité donnée."""
    if random.random() < taux_mutation:
        return voisin_swap(tournee)
    return tournee[:]


def selection_tournoi(population, couts, taille_tournoi=3):
    """Sélection par tournoi : choisit le meilleur parmi k individus aléatoires."""
    candidats = random.sample(range(len(population)), taille_tournoi)
    meilleur = min(candidats, key=lambda i: couts[i])
    return population[meilleur]


def algorithme_genetique(dist_matrix, n_clients, taille_pop=50, 
                         n_generations=500, taux_mutation=0.15,
                         taille_tournoi=3):
    """
    Algorithme Génétique pour le VRP/TSP.
    
    Paramètres :
        taille_pop : taille de la population
        n_generations : nombre de générations
        taux_mutation : probabilité de mutation
    """
    # Initialisation : population aléatoire
    population = []
    for _ in range(taille_pop):
        perm = list(range(1, n_clients + 1))
        random.shuffle(perm)
        population.append([0] + perm + [0])
    
    # Ajouter la solution PPV pour aider la convergence
    population[0] = plus_proche_voisin(dist_matrix)
    
    couts = [cout_tournee(ind, dist_matrix) for ind in population]
    
    meilleur_idx = np.argmin(couts)
    meilleure_tournee = population[meilleur_idx][:]
    meilleur_cout = couts[meilleur_idx]
    
    historique = [meilleur_cout]
    
    for gen in range(n_generations):
        nouvelle_pop = []
        
        # Élitisme : conserver le meilleur individu
        nouvelle_pop.append(meilleure_tournee[:])
        
        while len(nouvelle_pop) < taille_pop:
            # Sélection
            parent1 = selection_tournoi(population, couts, taille_tournoi)
            parent2 = selection_tournoi(population, couts, taille_tournoi)
            
            # Croisement OX
            enfant = croisement_ox(parent1, parent2)
            
            # Mutation
            enfant = mutation(enfant, taux_mutation)
            
            nouvelle_pop.append(enfant)
        
        population = nouvelle_pop
        couts = [cout_tournee(ind, dist_matrix) for ind in population]
        
        # Mise à jour du meilleur
        idx = np.argmin(couts)
        if couts[idx] < meilleur_cout:
            meilleure_tournee = population[idx][:]
            meilleur_cout = couts[idx]
        
        historique.append(meilleur_cout)
    
    return meilleure_tournee, meilleur_cout, historique


# Exécution
t0 = time.time()
tournee_ga, cout_ga, hist_ga = algorithme_genetique(dist_matrix, n_clients)
temps_ga = time.time() - t0

print(f"Algo. Génétique : coût = {cout_ga:.2f} (temps = {temps_ga:.3f}s)")
print(f"Amélioration vs PPV : {(cout_ppv - cout_ga) / cout_ppv * 100:.1f}%")
afficher_tournee(coords, tournee_ga, "Algorithme Génétique (OX + mutation)", cout_ga)

### 5.5 Comparaison visuelle des convergences

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(hist_hc, label=f'Hill Climbing (final: {cout_hc:.2f})', alpha=0.8)
ax.plot(hist_sa, label=f'Recuit Simulé (final: {cout_sa:.2f})', alpha=0.8)
ax.plot(hist_tabou, label=f'Recherche Tabou (final: {cout_tabou:.2f})', alpha=0.8)
ax.plot(hist_ga, label=f'Algo. Génétique (final: {cout_ga:.2f})', alpha=0.8)

ax.set_xlabel('Itérations / Générations', fontsize=12)
ax.set_ylabel('Coût de la meilleure solution', fontsize=12)
ax.set_title('Convergence des métaheuristiques — Instance à 20 clients', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
<a id='6'></a>
## 6. Génération d'instances et plan d'expérience

Comme le souligne Agathe, il est essentiel d'avoir un **plan d'expérience** structuré pour obtenir des résultats représentatifs et pouvoir tirer des conclusions valides.

### Plan d'expérience

**Facteurs à faire varier** :

| Facteur | Valeurs testées |
|---------|----------------|
| Taille de l'instance (n_clients) | 10, 20, 50 |
| Métaheuristique | HC, SA, Tabou, GA |
| Graine aléatoire | 5 graines différentes par config |

**Métriques** :
- Coût de la solution finale
- Temps d'exécution
- Écart relatif par rapport à la borne inférieure

In [ ]:
def borne_inferieure_mst(dist_matrix):
    """
    Borne inférieure basée sur l'arbre couvrant minimum (MST).
    Pour le TSP, le coût de l'arbre couvrant minimum est une borne inférieure
    car toute tournée hamiltonienne contient un arbre couvrant.
    """
    n = len(dist_matrix)
    # Algorithme de Prim
    dans_arbre = [False] * n
    cout_min = [float('inf')] * n
    cout_min[0] = 0
    cout_total = 0
    
    for _ in range(n):
        # Choisir le sommet non inclus avec le coût minimal
        u = -1
        for v in range(n):
            if not dans_arbre[v] and (u == -1 or cout_min[v] < cout_min[u]):
                u = v
        
        dans_arbre[u] = True
        cout_total += cout_min[u]
        
        # Mettre à jour les coûts des voisins
        for v in range(n):
            if not dans_arbre[v] and dist_matrix[u][v] < cout_min[v]:
                cout_min[v] = dist_matrix[u][v]
    
    return cout_total


# Test
bi = borne_inferieure_mst(dist_matrix)
print(f"Borne inférieure (MST) : {bi:.2f}")
print(f"Écart HC    : {(cout_hc - bi) / bi * 100:.1f}%")
print(f"Écart SA    : {(cout_sa - bi) / bi * 100:.1f}%")
print(f"Écart Tabou : {(cout_tabou - bi) / bi * 100:.1f}%")
print(f"Écart GA    : {(cout_ga - bi) / bi * 100:.1f}%")

### 6.1 Exécution du plan d'expérience

In [ ]:
# Plan d'expérience complet
tailles = [10, 20, 50]
graines = [42, 123, 456, 789, 1024]
resultats = []

for n_cl in tailles:
    for seed in graines:
        # Générer l'instance
        c, dm = generer_instance(n_cl, seed=seed)
        t_init = plus_proche_voisin(dm)
        bi = borne_inferieure_mst(dm)
        
        # Hill Climbing
        random.seed(seed)
        t0 = time.time()
        _, cout, _ = hill_climbing(dm, t_init, max_iter=5000)
        temps = time.time() - t0
        resultats.append({
            'n': n_cl, 'seed': seed, 'methode': 'Hill Climbing',
            'cout': cout, 'temps': temps, 'ecart_bi': (cout - bi) / bi * 100
        })
        
        # Recuit Simulé
        random.seed(seed)
        t0 = time.time()
        _, cout, _ = recuit_simule(dm, t_init, max_iter=10000)
        temps = time.time() - t0
        resultats.append({
            'n': n_cl, 'seed': seed, 'methode': 'Recuit Simulé',
            'cout': cout, 'temps': temps, 'ecart_bi': (cout - bi) / bi * 100
        })
        
        # Recherche Tabou
        random.seed(seed)
        t0 = time.time()
        _, cout, _ = recherche_tabou(dm, t_init, max_iter=2000)
        temps = time.time() - t0
        resultats.append({
            'n': n_cl, 'seed': seed, 'methode': 'Recherche Tabou',
            'cout': cout, 'temps': temps, 'ecart_bi': (cout - bi) / bi * 100
        })
        
        # Algorithme Génétique
        random.seed(seed)
        np.random.seed(seed)
        t0 = time.time()
        _, cout, _ = algorithme_genetique(dm, n_cl, taille_pop=50, n_generations=300)
        temps = time.time() - t0
        resultats.append({
            'n': n_cl, 'seed': seed, 'methode': 'Algo. Génétique',
            'cout': cout, 'temps': temps, 'ecart_bi': (cout - bi) / bi * 100
        })

print(f"Plan d'expérience terminé : {len(resultats)} expériences réalisées.")

---
<a id='7'></a>
## 7. Évaluation et comparaison des résultats

In [ ]:
# Agrégation des résultats
methodes = ['Hill Climbing', 'Recuit Simulé', 'Recherche Tabou', 'Algo. Génétique']

print("=" * 90)
print(f"{'Méthode':<20} {'n':>5} {'Coût moyen':>12} {'Écart BI (%)':>14} {'Temps moyen (s)':>16}")
print("=" * 90)

for n_cl in tailles:
    for meth in methodes:
        subset = [r for r in resultats if r['n'] == n_cl and r['methode'] == meth]
        cout_moy = np.mean([r['cout'] for r in subset])
        ecart_moy = np.mean([r['ecart_bi'] for r in subset])
        temps_moy = np.mean([r['temps'] for r in subset])
        print(f"{meth:<20} {n_cl:>5} {cout_moy:>12.2f} {ecart_moy:>13.1f}% {temps_moy:>15.3f}")
    print("-" * 90)

In [ ]:
# Visualisation : écart à la borne inférieure par méthode et taille
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

couleurs = {'Hill Climbing': '#e74c3c', 'Recuit Simulé': '#3498db', 
            'Recherche Tabou': '#2ecc71', 'Algo. Génétique': '#9b59b6'}

for idx, n_cl in enumerate(tailles):
    ax = axes[idx]
    data_par_methode = []
    labels = []
    colors = []
    
    for meth in methodes:
        subset = [r['ecart_bi'] for r in resultats if r['n'] == n_cl and r['methode'] == meth]
        data_par_methode.append(subset)
        labels.append(meth.replace(' ', '\n'))
        colors.append(couleurs[meth])
    
    bp = ax.boxplot(data_par_methode, labels=labels, patch_artist=True)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    
    ax.set_title(f'n = {n_cl} clients', fontsize=13)
    ax.set_ylabel('Écart à la borne inf. (%)' if idx == 0 else '')
    ax.grid(True, alpha=0.3)

fig.suptitle('Qualité des solutions par métaheuristique et taille d\'instance', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# Visualisation : temps d'exécution
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(tailles))
largeur = 0.2

for i, meth in enumerate(methodes):
    temps_moyens = []
    for n_cl in tailles:
        subset = [r['temps'] for r in resultats if r['n'] == n_cl and r['methode'] == meth]
        temps_moyens.append(np.mean(subset))
    ax.bar(x + i * largeur, temps_moyens, largeur, label=meth, 
           color=couleurs[meth], alpha=0.7)

ax.set_xlabel('Nombre de clients', fontsize=12)
ax.set_ylabel('Temps moyen (s)', fontsize=12)
ax.set_title('Temps d\'exécution par métaheuristique', fontsize=14)
ax.set_xticks(x + 1.5 * largeur)
ax.set_xticklabels(tailles)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

---
<a id='8'></a>
## 8. Analyse statistique et conclusions

In [ ]:
# Test statistique : les différences sont-elles significatives ?
from scipy import stats

print("Test de Kruskal-Wallis (non paramétrique) sur l'écart à la borne inférieure")
print("H0 : toutes les métaheuristiques ont les mêmes performances")
print("="*70)

for n_cl in tailles:
    groupes = []
    for meth in methodes:
        subset = [r['ecart_bi'] for r in resultats if r['n'] == n_cl and r['methode'] == meth]
        groupes.append(subset)
    
    stat, p_value = stats.kruskal(*groupes)
    print(f"\nn = {n_cl} clients : H = {stat:.4f}, p-value = {p_value:.4f}")
    if p_value < 0.05:
        print("  → Différence significative (p < 0.05) entre les métaheuristiques.")
    else:
        print("  → Pas de différence significative (p >= 0.05).")

### Synthèse et réponses aux questions du prosit

#### Pourquoi le Hill Climbing n'est pas une vraie métaheuristique ?

Le Hill Climbing ne dispose que d'un mécanisme d'**intensification** (il n'accepte que les améliorations). Il n'a aucune **diversification**, ce qui signifie qu'il se bloque au premier optimum local rencontré. Une métaheuristique se distingue justement par sa capacité à **échapper aux optima locaux** grâce à des mécanismes de diversification.

#### Comment les métaheuristiques gèrent l'équilibre intensification/diversification ?

| Métaheuristique | Intensification | Diversification |
|-----------------|----------------|------------------|
| **Recuit Simulé** | Accepte les améliorations | Accepte les dégradations avec proba $e^{-\Delta/T}$ décroissante |
| **Recherche Tabou** | Choisit le meilleur voisin | Liste tabou interdit le retour, force l'exploration |
| **Algo. Génétique** | Sélection des meilleurs | Mutation aléatoire + croisement |

#### Pourquoi le No Free Lunch impose de comparer ?

Le théorème NFL garantit qu'aucune métaheuristique n'est universellement supérieure. Pour un problème donné, les performances dépendent de la **structure du paysage de recherche** (rugosité, nombre d'optima locaux, etc.). Seule l'expérimentation permet de déterminer quelle approche est la plus adaptée.

#### Comment évaluer la qualité d'une solution ?

1. **Borne inférieure** : le coût de l'arbre couvrant minimum (MST) est une borne inférieure pour le TSP. L'écart entre la solution et cette borne donne une estimation de la qualité.
2. **Comparaison avec l'optimal** : si l'optimal est connu (méthodes exactes sur petites instances), on peut calculer l'écart exact.
3. **Tests statistiques** : répéter les expériences avec différentes graines et utiliser des tests (Kruskal-Wallis) pour vérifier la significativité des différences.

#### Importance du plan d'expérience

Un plan d'expérience structuré permet de :
- Faire varier les **paramètres des algorithmes** (température, taille tabou, taux de mutation…)
- Faire varier les **caractéristiques des instances** (taille, distribution spatiale…)
- Obtenir des résultats **statistiquement représentatifs** (plusieurs graines)
- Identifier les facteurs qui influencent le plus la qualité des solutions

---

## Bibliographie

- Faure, Lemaire et Picouleau, *Précis de recherche opérationnelle*, Dunod — Introduction et chapitre 11
- Moisdon et Nakhla, *Recherche opérationnelle*, Presses des Mines — Chapitre 8
- Siarry, *Métaheuristiques*, Eyrolles — Jusqu'à p.197
- Wolpert, D. H., & Macready, W. G. (1997). *No free lunch theorems for optimization*. IEEE Transactions on Evolutionary Computation.